# **TRAINING SCRIPT**
## *Chess Neural Network*

### **I - SETUP PHASE**

#### 1. Importing libraries

In [ ]:
import torch
import pickle
import numpy as np

from model import ChessCNN
from torch.utils.data import DataLoader, TensorDataset

#### 2. Defining global variables

On essaie de forcer les calculs effectués lors de l'entrainement sur le GPU en priorité. Ensuite nous initialisons les variables d'entrainement :
- Une taille de batch à 64 - *le modèle ne voit que 64 positions par 64 positions*
- Un nombre d'epochs à 100 - *l'entrainement va faire 100 passages sur le dataset complet*
- Un learning rate (taux d'apprentissage) à 0.0001 - *à chaque batch le réseau se rend compte de ses erreurs et va légèrement corriger*  

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-4

### **II - LOADING PHASE**

#### 1. Loading datasets

On charge en mémoire les données préparées par le script `Scripts/dataset.py`.

- `X.npy` : Il prend la forme d'un tableau NumPy (N, 13, 8, 8) où chaque élément est une position d'échecs encodée
- `y.npy` : Il prend la forme d'un tableau NumPy (N) où chaque valeur valeur est l'indice d'un coup joué en partie (stocké dans X)

En sortant les données de cette manière nos faisons en sorte que le modèle ne prédit pas les coups 'texte' mais les indices des coups.

Enfin `num_moves` recense le nombre total de coups uniques dans le dataset et va définir la taille de la dernière couche du réseau.

In [ ]:
X = np.load("../Data/Processed Database/X.npy")
y = np.load("../Data/Processed Database/y.npy")

with open("../Data/Processed Database/move_to_int.pkl", "rb") as f:
    move_to_int = pickle.load(f)

num_moves = len(move_to_int)

dataset = TensorDataset(torch.tensor(X),
                        torch.tensor(y))

#### 2. Loading model & optimizers

In [ ]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = ChessCNN(num_moves).to(DEVICE)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

### **III - TRAINING PHASE**

In [ ]:
for epoch in range(EPOCHS):

    model.train()
    total_loss = 0.0

    for xb, yb in loader:

        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)

        loss = criterion(logits, yb)
        loss.backward()

        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | loss={total_loss:.4f}")

torch.save(model.state_dict(), f"../Models/mdl_{BATCH_SIZE}_{EPOCHS}_{LR}.pth")
print("Model saved")